# Playground

In [1]:
import numpy as np
import pandas as pd

from utils.graph import (
    log_binned_density,
    plot_alpha_comparison,
    plot_alpha_estimates,
    plot_binning_comparison,
    plot_moments,
)

In [2]:
def p_x(alpha: float, k: float, x: float) -> float:
    return k * (x ** (-alpha))

def k_norm(alpha: float, x_min: float) -> float:
    return (alpha - 1) * (x_min ** (alpha - 1))

def cdf_x(alpha: float, x_min: float, x: float) -> float:
    return 1 - (x / x_min) ** (-(alpha - 1))

def mean_above(alpha: float, k: float, x: float) -> float:
    # Step 1: M(x) = k / (alpha - 2) * x^(2 - alpha)
    return k / (alpha - 2) * x ** (2 - alpha)

def sum_share_above(alpha: float, x_min: float, x: float) -> float:
    # Step 2: S(x) = M(x) / M(x_min)
    k = k_norm(alpha, x_min)
    return mean_above(alpha, k, x) / mean_above(alpha, k, x_min)

def x_top(alpha: float, x_min: float, q: float) -> float:
    # Step 3a: threshold x_q
    q = np.asarray(q, dtype=float)
    with np.errstate(divide="ignore"):  # q = 0 -> x_q = inf
        return x_min * q ** (-1 / (alpha - 1))

def top_share(alpha: float, q: float, x_min: float = 1.0) -> float:
    if alpha <= 2:
        # the mean diverges
        return np.ones_like(np.asarray(q, dtype=float))
    # Step 3b: S(x_q) = q^((alpha - 2) / (alpha - 1))
    return sum_share_above(alpha, x_min, x_top(alpha, x_min, q))

## Varying Alpha

In [3]:
ALPHAS = [1.5, 2.0, 2.16, 2.5, 3.0, 3.5]
X_MIN = 1.0
x = np.logspace(np.log10(X_MIN), 3, 500)

### PDF

$$p(x) = k \, x^{-\alpha}, \qquad k = (\alpha - 1)\, x_{\min}^{\alpha - 1}$$


$$\log p(x) = \log k - \alpha \log x$$

In [4]:
plot_alpha_comparison(
    ALPHAS,
    lambda alpha, x: p_x(alpha, k_norm(alpha, X_MIN), x),
    x,
    title=f"PDF  p(x) = k · x^(-α),  x_min = {X_MIN:g}",
    y_title="p(x)",
    log_y=True,
    linear_panel=True,
    linear_x_max=5 * X_MIN,
)

### CDF

$$F(x) = \int_{x_{\min}}^{x} k \, t^{-\alpha} \, dt = 1 - \left(\frac{x}{x_{\min}}\right)^{-(\alpha - 1)}$$

$$P(X > x) = 1 - F(x) = \left(\frac{x}{x_{\min}}\right)^{-(\alpha - 1)}$$

$$\log\left(1 - F(x)\right) = (\alpha - 1) \log x_{\min} - (\alpha - 1) \log x$$

In [5]:
plot_alpha_comparison(
    ALPHAS,
    lambda alpha, x: cdf_x(alpha, X_MIN, x),
    x,
    title=f"CDF  F(x) = 1 - (x / x_min)^(-(α - 1))  and CCDF  1 - F(x),  x_min = {X_MIN:g}",
    y_title="F(x)",
    log_log_curve=lambda alpha, x: 1 - cdf_x(alpha, X_MIN, x),
    log_log_y_title="1 - F(x)",
)

## Share of the total sum above x

In [6]:
q = np.linspace(1e-4, 1, 2000)

def share_or_one(alpha: float, q: np.ndarray) -> np.ndarray:
    if alpha <= 2:
        # the mean diverges
        return np.ones_like(q)
    return top_share(alpha, q, X_MIN)

plot_alpha_comparison(
    ALPHAS,
    share_or_one,
    q,
    title="Share of the total sum above x  S(x_q) = q^((α - 2) / (α - 1)),  q = 1 - F(x)",
    x_title="tail fraction of events  1 - F(x)",
    y_title="share of total sum above x",
    log_x=False,
    dashed=lambda alpha: alpha <= 2,
    label=lambda alpha: f"α = {alpha:g}" + ("  (mean diverges)" if alpha <= 2 else ""),
)

## Sampling

$$x = F^{-1}(u) = x_{\min} \, (1 - u)^{-1/(\alpha - 1)} , \quad u \sim U(0, 1)$$

In [7]:
SAMPLE_ALPHA = 2.16
SAMPLE_SIZES = [10**3, 10**5, 10**6, 10**8]
rng = np.random.default_rng(42)


def sample_power_law(alpha: float, x_min: float, n: int) -> np.ndarray:
    u = rng.uniform(size=n)
    return x_min * (1 - u) ** (-1 / (alpha - 1))

samples = {n: sample_power_law(SAMPLE_ALPHA, X_MIN, n) for n in SAMPLE_SIZES}

In [8]:
plot_binning_comparison(
    SAMPLE_ALPHA,
    X_MIN,
    pdf=lambda x: p_x(SAMPLE_ALPHA, k_norm(SAMPLE_ALPHA, X_MIN), x),
    ccdf=lambda x: 1 - cdf_x(SAMPLE_ALPHA, X_MIN, x),
    samples={n: samples[10**6][:n] for n in [10**3, 10**4, 10**5, 10**6]},
)

## Parameter Estimation

x_min is taken as the sample minimum, only α is estimated.

In [9]:
def alpha_lsq(centers: np.ndarray, density: np.ndarray) -> float:
    # minimise MSE of log10 p(x) against a line in log10 x
    slope, _ = np.polyfit(np.log10(centers), np.log10(density), deg=1)
    return -slope


def alpha_lsq_linear_bins(samples: np.ndarray, bins: int = 50) -> float:
    density, edges = np.histogram(samples, bins=bins, density=True)
    centers = (edges[:-1] + edges[1:]) / 2
    nonzero = density > 0  # log of an empty bin is -inf
    return alpha_lsq(centers[nonzero], density[nonzero])


def alpha_lsq_log_bins(samples: np.ndarray) -> float:
    return alpha_lsq(*log_binned_density(samples, x_min=samples.min()))

estimates = {
    "LSQ, linear bins": {n: alpha_lsq_linear_bins(s) for n, s in samples.items()},
    "LSQ, log bins": {n: alpha_lsq_log_bins(s) for n, s in samples.items()},
}
pd.DataFrame(estimates).rename_axis("N").round(3)

,"LSQ, linear bins","LSQ, log bins"
N,,
1000,1.407,2.135
100000,2.139,2.153
1000000,2.565,2.145
100000000,3.307,2.152


### Maximum likelihood

$$\hat\alpha = 1 + n \left[\sum_{i=1}^{n} \ln \frac{x_i}{x_{\min}}\right]^{-1}, \qquad \sigma = \sqrt{n} \left[\sum_{i=1}^{n} \ln \frac{x_i}{x_{\min}}\right]^{-1} = \frac{\hat\alpha - 1}{\sqrt{n}}$$

In [10]:
def alpha_mle(samples: np.ndarray) -> tuple[float, float]:
    n = len(samples)
    log_sum = np.log(samples / samples.min()).sum()
    alpha = 1 + n / log_sum
    sigma = (alpha - 1) / np.sqrt(n)
    return alpha, sigma

mle = {n: alpha_mle(s) for n, s in samples.items()}
estimates["MLE"] = {n: alpha for n, (alpha, _) in mle.items()}
errors = {"MLE": {n: sigma for n, (_, sigma) in mle.items()}}

pd.DataFrame(estimates).rename_axis("N").round(3)

,"LSQ, linear bins","LSQ, log bins",MLE
N,,,
1000,1.407,2.135,2.167
100000,2.139,2.153,2.157
1000000,2.565,2.145,2.161
100000000,3.307,2.152,2.160


In [11]:
plot_alpha_estimates(SAMPLE_ALPHA, {k: v for k, v in estimates.items() if k != "LSQ, linear bins"}, errors)

## Statisitical Moments

$$\text{mean} = x_{\min}\frac{\alpha - 1}{\alpha - 2}\ (\alpha > 2), \qquad \text{median} = x_{\min}\, 2^{1/(\alpha - 1)}, \qquad \text{std} = x_{\min}\sqrt{\frac{\alpha - 1}{\alpha - 3} - \left(\frac{\alpha - 1}{\alpha - 2}\right)^2}\ (\alpha > 3)$$

In [12]:
MOMENT_ALPHAS = [1.5, 2.16, 2.5, 3.5]
SAMPLE_SIZES = np.array([10**3, 10**4, 10**5, 10**6, 10**7, 10**8])

def true_moments(alpha: float, x_min: float) -> dict[str, float]:
    mean = x_min * (alpha - 1) / (alpha - 2) if alpha > 2 else np.inf
    std = x_min * np.sqrt((alpha - 1) / (alpha - 3) - ((alpha - 1) / (alpha - 2)) ** 2) if alpha > 3 else np.inf
    return {"mean": mean, "median": x_min * 2 ** (1 / (alpha - 1)), "std": std}

def running_moments(samples: np.ndarray, n: np.ndarray) -> dict[str, np.ndarray]:
    starts = np.concatenate([[0], n[:-1]])
    mean = np.cumsum(np.add.reduceat(samples[: n[-1]], starts)) / n
    mean_sq = np.cumsum(np.add.reduceat(samples[: n[-1]] ** 2, starts)) / n
    return {
        "mean": mean,
        "median": np.array([np.median(samples[:k]) for k in n]),
        "std": np.sqrt(mean_sq - mean**2),
    }

moments = {
    alpha: running_moments(sample_power_law(alpha, X_MIN, SAMPLE_SIZES[-1]), SAMPLE_SIZES)
    for alpha in MOMENT_ALPHAS
}

true_values = {alpha: true_moments(alpha, X_MIN) for alpha in MOMENT_ALPHAS}

In [13]:
plot_moments(SAMPLE_SIZES, moments, true_values, ALPHAS)